### **WE ADOPT THIS**  

In [1]:
import os
import numpy as np
import nibabel as nib
from scipy import ndimage as ndi

from rt_utils import RTStructBuilder

# -------------------------------------------------------------------
# 1. Hard-coded paths (adapt as needed)
# -------------------------------------------------------------------
# DICOM series folder (T2)
DICOM_SERIES_PATH = (
    "test-data/ProstateX-0004/"
    "10-18-2011-MR prostaat kanker detectie WDSmc MCAPRODETW-45493/"
    "5.000000-t2tsetra-75680"
)

# NIfTI tumour segmentation (T2 space)
NIFTI_MASK_PATH = (
    "PROSTATEx_masks/Files/lesions/Masks/T2/"
    "ProstateX-0004-Finding1-t2_tse_tra_ROI_hollow.nii.gz"
)

# Output RTSTRUCT file
OUTPUT_RTSTRUCT_PATH = "hollow_ProstateX-0004_RTSTRUCT.dcm"
# -------------------------------------------------------------------
# 1. Load NIfTI as a boolean mask in (z, y, x)
# -------------------------------------------------------------------
def load_nifti_mask(nifti_path: str) -> np.ndarray:
    """
    Load a NIfTI file and return a 3D boolean numpy array in (z, y, x) order,
    suitable for rt_utils.
    """
    nii = nib.load(nifti_path)
    data = nii.get_fdata()  # typically float or int

    # Create a binary mask: > 0 means “inside lesion”
    # dtype will be bool here
    mask = data > 0  

    mask = ndi.binary_fill_holes(mask) 

    # Nibabel gives (x, y, z); rt_utils expects slices on the last axis (rows, cols, slices)
    mask = np.transpose(mask, (1, 0, 2))  # (x,y,z) -> (y,x,z)
    # or simply skip transposing if (x,y,z) already matches your DICOM (rows, cols, slices=z)

    # RAS (NIfTI) -> LPS (DICOM) alignment:
    # flip the row axis so top/bottom line up with the DICOM viewer
    # ADDED THIS LINE TO FLIP THE MASK SO IT MATCHES THE DICOM VIEWER
    mask = mask[::-1, :, :]  # flip along rows (axis 0)

    # Optional debug:
    # print("Mask dtype:", mask.dtype)
    # print("Mask shape (z, y, x):", mask.shape)
    # print("Unique values:", np.unique(mask))

    return mask  # bool, 3D


# -------------------------------------------------------------------
# 2. Create RTSTRUCT
# -------------------------------------------------------------------
def create_rtstruct_from_mask(
    dicom_series_path: str,
    nifti_mask_path: str,
    output_rtstruct_path: str
) -> None:
    # Reference DICOM T2 series
    rtstruct = RTStructBuilder.create_new(dicom_series_path=dicom_series_path)

    # Load segmentation (boolean, (z,y,x))
    numpy_segmentation_mask = load_nifti_mask(nifti_mask_path)

    # Add as ROI (name and color as you like)
    rtstruct.add_roi(
        mask=numpy_segmentation_mask,
        name="Lesion_Finding1_T2",
        color=[255, 0, 0],  # optional
        use_pin_hole=True,
    )

    rtstruct.save(output_rtstruct_path)
    print("RTSTRUCT saved to:", os.path.abspath(output_rtstruct_path))

create_rtstruct_from_mask(
        dicom_series_path=DICOM_SERIES_PATH,
        nifti_mask_path=NIFTI_MASK_PATH,
        output_rtstruct_path=OUTPUT_RTSTRUCT_PATH,
    )


Writing file to hollow_ProstateX-0004_RTSTRUCT.dcm
RTSTRUCT saved to: /home/anson/work/prostate-mri-lesion-seg/hollow_ProstateX-0004_RTSTRUCT.dcm
